<a href="https://colab.research.google.com/github/drg799802/drg799802.github.io/blob/main/cheatgrass2026LRK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
# Removed geopandas import since it's not available
# from shapely.geometry import LineString

# Create sample data to replace the undefined variables
# 1. Sample habitat suitability matrix (2D array for heatmap)
habitat_suitability_matrix = np.random.rand(20, 20)  # 20x20 matrix with random values 0-1

# 2. Sample road network data (using simple coordinates instead of GeoDataFrame)
# Creating simple line coordinates to represent roads without geopandas
road_data = [
    {'x': [2, 8], 'y': [5, 12], 'road_id': 1},
    {'x': [1, 18], 'y': [15, 18], 'road_id': 2},
    {'x': [10, 15], 'y': [2, 19], 'road_id': 3}
]

# 3. Sample GBIF presence points (longitude and latitude coordinates)
np.random.seed(42)  # For reproducible random data
gbif_lon = np.random.uniform(0, 20, 15)  # 15 random longitude points
gbif_lat = np.random.uniform(0, 20, 15)  # 15 random latitude points

# Modified plotting code without geopandas
fig, ax = plt.subplots(figsize=(10, 8))

# 1. Background heatmap for PRISM Climate/Habitat Suitability (Continuous Palette)
# Using 'YlOrRd' (Yellow-Orange-Red) to indicate suitability intensity
sns.heatmap(habitat_suitability_matrix, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Invasion Suitability'})

# 2. Plotting the road network disturbance zones using matplotlib instead of geopandas
# Plot each road as a line using matplotlib's plot function
for road in road_data:
    ax.plot(road['x'], road['y'], color='black', linewidth=1.5, label='Disturbance Buffers (Roads)' if road['road_id'] == 1 else "")

# 3. Plotting ground-truth presence points
# Bright magenta or cyan dots will pop cleanly against a Yellow-Orange-Red background
plt.scatter(gbif_lon, gbif_lat, color='magenta', s=10, label='GBIF Verified Presence')

plt.title("Cheatgrass Containment Mapping: Suitability vs. Infrastructure")
plt.legend()
plt.show()

import ee
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. INITIALIZE EARTH ENGINE
# ==========================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# ==========================================
# 2. DEFINE BOUNDARIES & CONFIGURATION
# ==========================================
# Extract the Pine Ridge Reservation boundary using Census TIGER data
reservations = ee.FeatureCollection("TIGER/2018/LandAndWater/AmericanIndianAreaRCRSA")
pine_ridge = reservations.filter(ee.Filter.eq('NAME', 'Pine Ridge'))

# Parameters
START_YEAR = 2015
END_YEAR = 2025
INVADED_THRESHOLD = 15  # Percent canopy cover to count a pixel as "highly invaded"
YEARS = list(range(START_YEAR, END_YEAR + 1))

# Load the RAP Vegetation Cover ImageCollection (V3 or latest)
# 'AFG' stands for Annual Forbs and Grasses (the functional group containing Cheatgrass)
rap_collection = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-v2")

# ==========================================
# 3. ANALYSIS PIPELINE
# ==========================================
data_summary = []

print("Analyzing Earth Engine satellite data layers...")

for year in YEARS:
    # Filter dataset for specific year and select the AFG band
    rap_year = rap_collection.filter(ee.Filter.calendarRange(year, year, 'year')).first()
    afg_cover = rap_year.select('AFG').clip(pine_ridge)

    # Calculate Mean Cover across the entire reservation
    mean_dict = afg_cover.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pine_ridge.geometry(),
        scale=30,
        maxPixels=1e9
    )
    mean_cover = mean_dict.get('AFG').getInfo()

    # Calculate total area exceeding the invasion threshold (>15% cover)
    invaded_mask = afg_cover.gte(INVADED_THRESHOLD)
    area_image = invaded_mask.multiply(ee.Image.pixelArea())  # Area in square meters

    area_dict = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=pine_ridge.geometry(),
        scale=30,
        maxPixels=1e9
    )
    # Convert square meters to acres (1 sq meter = 0.000247105 acres)
    invaded_acres = area_dict.get('AFG').getInfo() * 0.000247105

    data_summary.append({
        'Year': year,
        'Mean_Cover_Pct': mean_cover,
        'Invaded_Acres': invaded_acres
    })
    print(f"✔ Processed Year: {year}")

# Convert findings to Pandas DataFrame
df_trends = pd.DataFrame(data_summary)

# ==========================================
# 4. DATA VISUALIZATION
# ==========================================
sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot 1: Mean Cover Trend (Line)
color = '#d95f02'
ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean AFG Cover (%)', color=color, fontsize=12, fontweight='bold')
line = ax1.plot(df_trends['Year'], df_trends['Mean_Cover_Pct'], color=color, marker='o', linewidth=2.5, label='Mean Cover %')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(YEARS)

# Plot 2: Total Invaded Acreage (Bars)
ax2 = ax1.twinx()
color = '#7570b3'
ax2.set_ylabel(f'Total Acres Exceeding {INVADED_THRESHOLD}% Cover', color=color, fontsize=12, fontweight='bold')
bars = ax2.bar(df_trends['Year'], df_trends['Invaded_Acres'], color=color, alpha=0.3, width=0.4, label='Invaded Acreage')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f"Pine Ridge Reservation: Cheatgrass (AFG) Spread Trends ({START_YEAR}-{END_YEAR})", fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

# Display the raw statistical metrics
print("\n--- Summary Statistics Table ---")
print(df_trends.to_string(index=False))

## Summary Statistics for Filtered GBIF Occurrence Data

Let's examine the summary statistics for the cleaned and filtered GBIF occurrence data to understand its distribution and characteristics.

In [ ]:
print("--- Descriptive Statistics ---")
display(df_final.describe())

print("\n--- DataFrame Information (Data Types and Non-Null Counts) ---")
df_final.info()

print("\n--- Unique Values for Key Categorical/Temporal Columns ---")
for col in ['species', 'countryCode', 'stateProvince', 'year', 'month', 'day']:
    if col in df_final.columns:
        print(f"\nUnique values for '{col}':")
        display(df_final[col].value_counts().head(10)) # Display top 10 unique values


## Summary of GBIF Occurrence Data Processing

This section of the notebook retrieves and processes occurrence data for *Bromus tectorum* (Cheatgrass) from the Global Biodiversity Information Facility (GBIF). The key steps include:

1.  **Taxon Key Retrieval**: Automatically fetches the unique GBIF taxon key for Cheatgrass, with a fallback ID to ensure robustness.
2.  **Paginated Data Download**: Downloads a specified number of Cheatgrass occurrence records from GBIF, with a focus on records within the United States and those with valid coordinate information. A paging mechanism is used to handle large datasets efficiently.
3.  **Initial Data Cleaning & Filtering**: Removes records with missing or zero-value latitude/longitude, ensuring data quality for geospatial analysis. It then selects a relevant subset of columns for further analysis.
4.  **Geospatial Conversion & Filtering**: Converts the cleaned DataFrame into a GeoDataFrame, which is a specialized DataFrame for geospatial data. It initially filters for records within South Dakota (relevant to the Pine Ridge Reservation context) and then reprojects the coordinates to an Albers Equal Area Conic projection (EPSG:5070), which is suitable for accurate area calculations.
5.  **Interactive Map Render**: Generates an interactive Folium map displaying the first 500 Cheatgrass occurrence points. The map is centered on the average coordinates of the collected data and uses Esri World Imagery as a base layer. The output is saved as an HTML file and can also be displayed inline.

## Summary of Phenological Analysis for Cheatgrass Detection (MODIS Data)

This section outlines a method for detecting Cheatgrass using MODIS NDVI data by leveraging its unique phenological characteristics. The core idea is to identify areas where vegetation shows early green-up in spring followed by a significant decline in summer, a signature characteristic of Cheatgrass.

1.  **Configuration & Parameters**: Sets up paths to your AppEEARS unpacked MODIS NDVI GeoTIFF rasters and defines specific Day of Year (DOY) files for early spring (e.g., late March/April) and early summer (e.g., June).
2.  **Load and Scale NDVI**: A helper function `load_and_scale_ndvi` is used to load MODIS NDVI GeoTIFFs. It automatically applies the MODIS scaling factor (0.0001) to convert raw values into true NDVI values ranging from -1.0 to 1.0, and handles no-data values.
3.  **Phenological Analysis Pipeline**:
    *   **Calculate Phenological Cheatgrass Proxy Index**: Computes the difference between Spring NDVI and Summer NDVI (`spring_ndvi - summer_ndvi`). A positive and substantial difference indicates early peak productivity followed by rapid senescence, which is typical for Cheatgrass.
    *   **Apply Threshold Filters**: Defines a mask based on two criteria:
        *   High spring productivity (e.g., Spring NDVI > 0.35).
        *   Significant drop into summer (e.g., `pheno_diff` > 0.15).
    *   **Clear Nodata Values**: Ensures that areas with no data in either spring or summer NDVI are not incorrectly classified.
4.  **Export & Visualization**:
    *   **Export Mask**: Creates and exports a binary GeoTIFF raster (`cheatgrass_pheno_proxy.tif`) where 1 indicates potential Cheatgrass presence and 0 indicates other vegetation or non-cheatgrass areas.
    *   **Plotting**: Visualizes both the raw phenological difference (Spring NDVI - Summer NDVI) using a diverging colormap and the final isolated cheatgrass footprint (the binary mask). These visualizations help in understanding the spatial distribution of the detected Cheatgrass signature.

This framework highlights how specific phenological patterns, derived from satellite imagery, can serve as a powerful proxy for identifying invasive species like Cheatgrass.

# ==========================================
# 1. INITIALIZE EARTH ENGINE
# ==========================================
import ee
try:
    # !!! REPLACE 'your-gcp-project-id' with your actual Google Cloud Project Name/ID !!!
    ee.Initialize(project='your-gcp-project-id')
except Exception as e:
    print("Authentication required or project missing. Requesting access token...")
    ee.Authenticate()
    # !!! REPLACE HERE AS WELL !!!
    ee.Initialize(project='your-gcp-project-id')

# ==========================================
# 2. DEFINE BOUNDARIES & CONFIGURATION
# ==========================================
# Extract the Pine Ridge Reservation boundary using Census TIGER data
reservations = ee.FeatureCollection("TIGER/2018/LandAndWater/AmericanIndianAreaRCRSA")
pine_ridge = reservations.filter(ee.Filter.eq('NAME', 'Pine Ridge'))

# Parameters
START_YEAR = 2015
END_YEAR = 2025
INVADED_THRESHOLD = 15  # Percent canopy cover to count a pixel as "highly invaded"
YEARS = list(range(START_YEAR, END_YEAR + 1))

# FIXED: Swapped out the legacy 'rap-data-365417' path for the official V3 collection
rap_collection = ee.ImageCollection("projects/rangeland-analysis-platform/assets/vegetation-cover-v3")

import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

# =========================================================================
# CONFIGURATION & PARAMETERS
# =========================================================================
# Folder containing your AppEEARS unpacked MODIS NDVI GeoTIFF rasters
TIFF_DIR = "path_to_your_appeears_outputs"

# Define DOY (Day of Year) files for your target phenological windows
# Early Spring: Cheatgrass rapid green-up (e.g., late March/April, DOY ~081 to 121)
# Early Summer: Cheatgrass senesces/turns brown while perennials green up (DOY ~161+)
SPRING_TIFF = os.path.join(TIFF_DIR, "*A2025097*.tif") # Example: DOY 097 (April 7)
SUMMER_TIFF = os.path.join(TIFF_DIR, "*A2025161*.tif") # Example: DOY 161 (June 10)

def load_and_scale_ndvi(file_pattern):
    """Finds a file matching the pattern, reads it, and applies MODIS scaling factors."""
    match = glob.glob(file_pattern)
    if not match:
        raise FileNotFoundError(f"No file found matching pattern: {file_pattern}")
    
    with rasterio.open(match[0]) as src:
        # Read first band
        ndvi = src.read(1).astype(np.float32)
        # MODIS NDVI valid range is -2000 to 10000; Scale factor is 0.0001
        ndvi = np.where((ndvi >= -2000) & (ndvi <= 10000), ndvi * 0.0001, np.nan)
        profile = src.profile
    return ndvi, profile

# =========================================================================
# PHENOLOGICAL ANALYSIS PIPELINE
# =========================================================================
print("Loading AppEEARS foundational satellite layers...")
spring_ndvi, img_profile = load_and_scale_ndvi(SPRING_TIFF)
summer_ndvi, _ = load_and_scale_ndvi(SUMMER_TIFF)

# 1. Calculate Phenological Cheatgrass Proxy Index
# Cheatgrass reaches high NDVI early, then craters dramatically as it dries out (senesces)
# While native perennials maintain or increase their NDVI during early summer.
pheno_diff = spring_ndvi - summer_ndvi

# 2. Apply Threshold Filters
# - High spring productivity (NDVI > 0.35)
# - Significant drop into summer (Difference > 0.15)
cheatgrass_mask = (spring_ndvi > 0.35) & (pheno_diff > 0.15)

# Clear up nodata values
cheatgrass_mask = np.where(np.isnan(spring_ndvi) | np.isnan(summer_ndvi), 0, cheatgrass_mask)

# =========================================================================
# EXPORT & VISUALIZATION
# =========================================================================
# Update metadata profile to output a binary mask raster (1 = Cheatgrass, 0 = Other)
img_profile.update(dtype=rasterio.uint8, count=1, nodata=0)

output_mask_path = os.path.join(TIFF_DIR, "cheatgrass_pheno_proxy.tif")
with rasterio.open(output_mask_path, "w", **img_profile) as dst:
    dst.write(cheatgrass_mask.astype(rasterio.uint8), 1)
print(f"✔ Phenological proxy raster exported to: {output_mask_path}")

# Plotting the raw signature mapping
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left plot: The phenological difference raw value
im1 = ax1.imshow(pheno_diff, cmap="bwr", vmin=-0.4, vmax=0.4)
fig.colorbar(im1, ax=ax1, label="Spring NDVI minus Summer NDVI")
ax1.set_title("Phenological Shift (Positive = Early Peak)")

# Right plot: The isolated proxy footprint
ax2.imshow(cheatgrass_mask, cmap="Greens")
ax2.set_title("Surrogated Cheatgrass Footprint (Threshold Matrix)")

plt.tight_layout()
plt.show()

Key Logic Behind This Framework:MODIS Data Clean-up: AppEEARS drops the raw HDF datasets into clean GeoTIFFs, but they are still scaled by $10,000$. The script handles the 0.0001 scale transformation automatically to give you true baseline NDVI values between $-1.0$ and $1.0$.Phenological Core Matrix: By calculating $NDVI_{Spring} - NDVI_{Summer}$, you capture the exact ecological niche of cheatgrass. Native vegetation or perennial rangeland crops will return negative or near-zero changes over this specific spring-to-summer transition window.


## Overlaying GBIF Occurrences on Pine Ridge Cheatgrass Suitability Map

This section generates an interactive Folium map, overlaying the GBIF verified Cheatgrass presence points (`gdf_sd`) onto the phenological suitability mask (`cheatgrass_pheno_proxy.tif`) that was generated in the previous step. This visualization allows for a direct comparison between observed occurrences and areas identified as highly suitable for Cheatgrass invasion based on MODIS NDVI phenology.

**Prerequisites:**
- Ensure the phenological analysis cell directly above has been successfully executed.
- Verify that the `TIFF_DIR` variable is correctly set to the directory containing your AppEEARS unpacked MODIS outputs, where `cheatgrass_pheno_proxy.tif` is saved.

In [ ]:
import folium
import rasterio
import numpy as np
import os
import pyproj

# --- User Configuration ---
# IMPORTANT: Set this to the actual path where 'cheatgrass_pheno_proxy.tif' is located.
# This directory must be the SAME as the `TIFF_DIR` used in the phenological analysis cell (`R90rBkulGNtq`).
# Example: TIFF_DIR = '/content/cheatgrass_outputs/'
TIFF_DIR = "/content/cheatgrass_outputs" # <--- PLEASE UPDATE THIS PATH WITH YOUR ACTUAL DIRECTORY

# Construct the full path to the generated cheatgrass mask GeoTIFF
output_mask_path = os.path.join(TIFF_DIR, "cheatgrass_pheno_proxy.tif")

if not os.path.exists(output_mask_path):
    print(f"Error: Cheatgrass pheno proxy file not found at {output_mask_path}")
    print("\n--- ACTION REQUIRED ---")
    print("Please ensure the phenological analysis cell (`R90rBkulGNtq`) was run successfully ")
    print(f"and that its `TIFF_DIR` is set to a writable path like '{TIFF_DIR}'.")
    print("Once the file is generated, ensure this cell's `TIFF_DIR` matches and re-run.")
else:
    print(f"Loading cheatgrass suitability mask from: {output_mask_path}")
    # --- Load Raster Data and Transform Bounds ---
    with rasterio.open(output_mask_path) as src:
        mask_data = src.read(1) # Read the first band
        src_bounds = src.bounds
        src_crs = src.crs

        # Define the target CRS (WGS84 lat/lon)
        target_crs = pyproj.CRS("EPSG:4326")

        # Create a transformer to convert bounds to lat/lon
        transformer = pyproj.Transformer.from_crs(src_crs, target_crs, always_xy=True)

        # Transform the corner coordinates of the raster
        min_lon, min_lat = transformer.transform(src_bounds.left, src_bounds.bottom)
        max_lon, max_lat = transformer.transform(src_bounds.right, src_bounds.top)

        # Folium's ImageOverlay expects bounds as [[south, west], [north, east]]
        image_bounds_latlon = [[min_lat, min_lon], [max_lat, max_lon]]

    # Normalize mask_data for better visualization if it's not already 0-1
    # The cheatgrass_mask is binary (0 or 1), so no normalization needed for 0-1 range.

    # --- Map Initialization ---
    # Calculate map center from the transformed raster bounds
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2

    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="OpenStreetMap")

    # --- Add Cheatgrass Suitability Mask as ImageOverlay ---
    # Using a custom colormap for the binary mask (e.g., transparent for 0, green for 1)
    # Or, as in the previous plot, just use a Greens colormap.
    # For a binary mask, a simple ImageOverlay often works well directly.
    folium.raster_layers.ImageOverlay(
        image=mask_data, # The numpy array of the mask
        bounds=image_bounds_latlon, # Transformed lat/lon bounds
        colormap=lambda x: '#006400' if x > 0 else '#FFFFFF00', # Green for cheatgrass, transparent for others
        opacity=0.6, # Adjust transparency
        name='Cheatgrass Suitability (Pheno Proxy)'
    ).add_to(m)

    # --- Add GBIF Occurrence Points (gdf_sd) ---
    if 'gdf_sd' in locals() and not gdf_sd.empty:
        # Iterate through a sample of points for performance if gdf_sd is very large
        # For now, let's plot all of them if not too many.
        print(f"Adding {len(gdf_sd)} GBIF occurrence points...")
        for idx, row in gdf_sd.iterrows():
            # Ensure geometry is valid and in expected format
            if row.geometry and row.geometry.geom_type == 'Point':
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x], # Latitude, Longitude
                    radius=3,
                    color='red',
                    fill=True,
                    fill_color='red',
                    fill_opacity=0.8,
                    popup=f"Year: {row.get('year', 'N/A')}<br>Locality: {row.get('locality', 'N/A')}"
                ).add_to(m)
        print("GBIF points added.")
    else:
        print("No South Dakota GBIF occurrence data (gdf_sd) found or it is empty. Skipping points overlay.")

    # Add a Layer Control to toggle layers on/off
    folium.LayerControl().add_to(m)

    print("\nInteractive map created.")
    # Display the map
    display(m)

In [ ]:
import os

output_dir = "/content/cheatgrass_outputs"
if os.path.isdir(output_dir):
    print(f"The directory '{output_dir}' exists.")
    print("Please ensure the phenological analysis cell (`R90rBkulGNtq`) was run and saved the .tif file there.")
else:
    print(f"The directory '{output_dir}' does NOT exist. Please create it first and ensure `TIFF_DIR` in cell `R90rBkulGNtq` points to it, then run `R90rBkulGNtq`.")
    # Optionally, create the directory if it doesn't exist and the user agrees
    # os.makedirs(output_dir, exist_ok=True)
    # print(f"Created directory: {output_dir}")

In [ ]:
import os

directory_to_create = "/content/cheatgrass_outputs"

# Create the directory if it doesn't exist
os.makedirs(directory_to_create, exist_ok=True)

print(f"Directory '{directory_to_create}' ensured to exist.")
print("Now, please run the phenological analysis cell (`R90rBkulGNtq`) to generate the .tif file into this directory,")
print("and then re-run the map overlay cell (`b22c8268`).")